# MS2 Road3 Metric Depth Evaluation

This notebook evaluates five input pipelines on a uniformly sampled set of 200 MS2 Road3 left-camera images:

- Original
- Gamma correction
- CLAHE
- Multi-Scale Retinex (MSR)
- LLFormer

DepthAnythingV2 Metric Outdoor Small is used for metric-depth prediction. Evaluation is performed against the filtered MS2 reference depth maps using MAE, RMSE, and AbsRel.

A consistent valid-depth range of **0–80 m** is used throughout the notebook. No blending experiment is included.


## 1. Environment Setup


In [1]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path
from collections import OrderedDict

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm import tqdm
from skimage import img_as_ubyte

WORK_DIR = Path("/kaggle/working")
LLFORMER_REPO = WORK_DIR / "LLFormer"


def run_command(command, cwd=None):
    print(f"\nRunning: {command}")
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n{command}"
        )


if not LLFORMER_REPO.is_dir():
    run_command(
        "git clone https://github.com/TaoWangzj/LLFormer.git",
        cwd=WORK_DIR
    )
else:
    print(f"LLFormer already exists: {LLFORMER_REPO}")


warmup_dir = LLFORMER_REPO / "pytorch-gradual-warmup-lr"

if warmup_dir.is_dir():
    run_command(
        f'"{sys.executable}" setup.py install',
        cwd=warmup_dir
    )
else:
    run_command(
        f'"{sys.executable}" -m pip install '
        f'pytorch-gradual-warmup-lr -q'
    )


run_command(
    f'"{sys.executable}" -m pip install '
    f'natsort yacs gdown transformers scikit-image -q'
)

if str(LLFORMER_REPO) not in sys.path:
    sys.path.insert(0, str(LLFORMER_REPO))

print("\nEnvironment setup completed.")



Running: git clone https://github.com/TaoWangzj/LLFormer.git
Cloning into 'LLFormer'...


Running: "/usr/bin/python3" setup.py install
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
running build
running build_py
creating build/lib/warmup_scheduler
copying warmup_scheduler/run.py -> build/lib/warmup_scheduler


## 2. Download and Locate LLFormer LOL Weights


In [2]:
LLFORMER_CHECKPOINT_DIR = (
    LLFORMER_REPO / "checkpoints" / "LOL"
)
LLFORMER_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def find_llformer_checkpoint(root_dir):
    candidates = list(
        root_dir.rglob("model_bestPSNR.pth")
    )

    if not candidates:
        return None

    candidates.sort(
        key=lambda path: (
            "models" not in path.parts,
            len(path.parts)
        )
    )
    return candidates[0]


weights_path = find_llformer_checkpoint(
    LLFORMER_CHECKPOINT_DIR
)

if weights_path is None:
    run_command(
        f'"{sys.executable}" -m gdown --folder '
        f'"https://drive.google.com/drive/folders/'
        f'1J7NvvPsCtT0j8Rd9ombJ6sVIC6v0Xweb" '
        f'-O "{LLFORMER_CHECKPOINT_DIR}"'
    )

    weights_path = find_llformer_checkpoint(
        LLFORMER_CHECKPOINT_DIR
    )


if weights_path is None:
    raise FileNotFoundError(
        "model_bestPSNR.pth could not be found "
        "under the LLFormer checkpoint directory."
    )


print("LLFormer checkpoint ready:")
print(weights_path)



Running: "/usr/bin/python3" -m gdown --folder "https://drive.google.com/drive/folders/1J7NvvPsCtT0j8Rd9ombJ6sVIC6v0Xweb" -O "/kaggle/working/LLFormer/checkpoints/LOL"
Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1nTLY_gChn2_Va2BfS_R_W-bdjsqsLHJB
From (redirected): https://drive.google.com/uc?id=1nTLY_gChn2_Va2BfS_R_W-bdjsqsLHJB&confirm=t&uuid=dcc2c579-a4d5-4510-a3bf-d7b8cd34df5b
To: /kaggle/working/LLFormer/checkpoints/LOL/models/model_bestPSNR.pth
Retrieving folder 1AZchyGAR5vtQpIEZTdDYH9t605nk_YBR models
Processing file 1nTLY_gChn2_Va2BfS_R_W-bdjsqsLHJB model_bestPSNR.pth
Processing file 1S7FlcG-aEMUfDEsh9ZdcMnc5G0sfE4MB model_bestSSIM.pth

  0%|          | 0.00/296M [00:00<?, ?B/s]
  1%|          | 2.62M/296M [00:00<00:16, 18.3MB/s]
  2%|▏         | 5.24M/296M [00:00<00:23, 12.1MB/s]
  7%|▋         | 19.4M/296M [00:00<00:05, 48.3MB/s]

## 3. Dataset Paths and Uniform Sampling


In [3]:
RGB_DIR = Path(
    "/kaggle/input/datasets/mavislei/"
    "ms2-road3/full/MS2_Road3/rgb/img_left"
)

GT_DIR = Path(
    "/kaggle/input/datasets/mavislei/"
    "ms2-road3/full/MS2_Road3/depth_filtered"
)

if not RGB_DIR.is_dir():
    raise FileNotFoundError(
        f"RGB directory not found: {RGB_DIR}"
    )

if not GT_DIR.is_dir():
    raise FileNotFoundError(
        f"GT directory not found: {GT_DIR}"
    )


rgb_files = sorted(
    f.name for f in RGB_DIR.iterdir()
    if f.suffix.lower() == ".png"
)

gt_files = {
    f.name for f in GT_DIR.iterdir()
    if f.suffix.lower() == ".png"
}

common_files = [
    filename for filename in rgb_files
    if filename in gt_files
]

NUM_SAMPLES = 200

if len(common_files) < NUM_SAMPLES:
    raise ValueError(
        f"Only {len(common_files)} matched RGB/GT files "
        f"were found, fewer than {NUM_SAMPLES}."
    )

sample_indices = np.linspace(
    0,
    len(common_files) - 1,
    NUM_SAMPLES,
    dtype=int
)

sample_files = [
    common_files[index]
    for index in sample_indices
]

sample_csv = (
    WORK_DIR / "MS2_Road3_sample_200.csv"
)

pd.DataFrame({
    "filename": sample_files
}).to_csv(
    sample_csv,
    index=False
)

print(f"Matched RGB/GT images: {len(common_files)}")
print(f"Uniformly sampled images: {len(sample_files)}")
print(f"Sample list saved to: {sample_csv}")
print("First five samples:", sample_files[:5])


Matched RGB/GT images: 2539
Uniformly sampled images: 200
Sample list saved to: /kaggle/working/MS2_Road3_sample_200.csv
First five samples: ['000000.png', '000012.png', '000025.png', '000038.png', '000051.png']


## 4. Define Enhancement and Utility Functions


In [4]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


def apply_gamma(image_bgr, gamma=2.2):
    inv_gamma = 1.0 / gamma

    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255.0
        for i in range(256)
    ]).astype(np.uint8)

    return cv2.LUT(image_bgr, table)


def apply_clahe(
    image_bgr,
    clip_limit=2.0,
    tile_grid_size=(8, 8)
):
    lab = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )

    l_enhanced = clahe.apply(l_channel)

    enhanced_lab = cv2.merge([
        l_enhanced,
        a_channel,
        b_channel
    ])

    return cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )


def apply_msr(
    image_bgr,
    sigmas=(15, 80, 250)
):
    image = image_bgr.astype(np.float32) + 1.0
    retinex = np.zeros_like(
        image,
        dtype=np.float32
    )

    for sigma in sigmas:
        blurred = cv2.GaussianBlur(
            image,
            (0, 0),
            sigma
        )

        retinex += (
            np.log(image)
            - np.log(blurred + 1e-6)
        )

    retinex /= len(sigmas)

    for channel_index in range(3):
        channel = retinex[:, :, channel_index]
        channel_min = channel.min()
        channel_max = channel.max()

        retinex[:, :, channel_index] = (
            (channel - channel_min)
            / (channel_max - channel_min + 1e-8)
            * 255.0
        )

    return np.clip(
        retinex,
        0,
        255
    ).astype(np.uint8)


def depth_to_vis(depth):
    depth = np.asarray(
        depth,
        dtype=np.float32
    )

    valid = np.isfinite(depth)

    if valid.sum() == 0:
        return np.zeros(
            (*depth.shape, 3),
            dtype=np.uint8
        )

    depth_min = depth[valid].min()
    depth_max = depth[valid].max()

    normalised = np.zeros_like(
        depth,
        dtype=np.float32
    )

    normalised[valid] = (
        depth[valid] - depth_min
    ) / (
        depth_max - depth_min + 1e-8
    )

    depth_uint8 = np.clip(
        normalised * 255.0,
        0,
        255
    ).astype(np.uint8)

    return cv2.applyColorMap(
        depth_uint8,
        cv2.COLORMAP_TURBO
    )


def load_reference_depth(gt_path):
    gt_raw = cv2.imread(
        str(gt_path),
        cv2.IMREAD_UNCHANGED
    )

    if gt_raw is None:
        raise FileNotFoundError(
            f"Unable to read reference depth: {gt_path}"
        )

    return (
        gt_raw.astype(np.float32) / 256.0
    )


def compute_metrics(
    prediction,
    reference_depth,
    max_depth=80.0
):
    prediction = np.asarray(
        prediction,
        dtype=np.float32
    )

    reference_depth = np.asarray(
        reference_depth,
        dtype=np.float32
    )

    if prediction.shape != reference_depth.shape:
        prediction = cv2.resize(
            prediction,
            (
                reference_depth.shape[1],
                reference_depth.shape[0]
            ),
            interpolation=cv2.INTER_CUBIC
        )

    valid_mask = (
        (reference_depth > 0.0)
        & (reference_depth <= max_depth)
        & np.isfinite(reference_depth)
        & np.isfinite(prediction)
        & (prediction > 0.0)
    )

    valid_pixels = int(valid_mask.sum())

    if valid_pixels == 0:
        return None

    pred_valid = prediction[valid_mask]
    gt_valid = reference_depth[valid_mask]

    mae = float(
        np.mean(
            np.abs(pred_valid - gt_valid)
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                (pred_valid - gt_valid) ** 2
            )
        )
    )

    absrel = float(
        np.mean(
            np.abs(pred_valid - gt_valid)
            / gt_valid
        )
    )

    return {
        "mae": mae,
        "rmse": rmse,
        "absrel": absrel,
        "valid_pixels": valid_pixels
    }


print("Enhancement and utility functions are ready.")


Enhancement and utility functions are ready.


## 5. Pre-compute LLFormer Images

LLFormer is loaded first, used to enhance all 200 sampled images, and then unloaded before DepthAnythingV2 is loaded. This reduces GPU memory usage.


In [5]:
from model.LLFormer import LLFormer as LLFormerModel


def load_llformer():
    model = LLFormerModel(
        inp_channels=3,
        out_channels=3,
        dim=16,
        num_blocks=[2, 4, 8, 16],
        num_refinement_blocks=2,
        heads=[1, 2, 4, 8],
        ffn_expansion_factor=2.66,
        bias=False,
        LayerNorm_type="WithBias",
        attention=True,
        skip=False
    )

    checkpoint = torch.load(
        str(weights_path),
        map_location="cpu"
    )

    state_dict = checkpoint["state_dict"]

    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        cleaned_state_dict = OrderedDict()

        for key, value in state_dict.items():
            cleaned_key = (
                key[7:]
                if key.startswith("module.")
                else key
            )

            cleaned_state_dict[
                cleaned_key
            ] = value

        model.load_state_dict(
            cleaned_state_dict
        )

    return model.to(DEVICE).eval()


def apply_llformer(
    image_bgr,
    llformer_model,
    max_size=512
):
    original_height, original_width = (
        image_bgr.shape[:2]
    )

    resized = image_bgr
    height, width = resized.shape[:2]

    if max(height, width) > max_size:
        scale = max_size / max(
            height,
            width
        )

        resized = cv2.resize(
            resized,
            (
                max(1, int(round(width * scale))),
                max(1, int(round(height * scale)))
            ),
            interpolation=cv2.INTER_AREA
        )

    image_rgb = cv2.cvtColor(
        resized,
        cv2.COLOR_BGR2RGB
    )

    input_tensor = TF.to_tensor(
        Image.fromarray(image_rgb)
    ).unsqueeze(0).to(DEVICE)

    height, width = input_tensor.shape[2:]
    multiple = 16

    padded_height = (
        (height + multiple - 1) // multiple
    ) * multiple

    padded_width = (
        (width + multiple - 1) // multiple
    ) * multiple

    input_tensor = F.pad(
        input_tensor,
        (
            0,
            padded_width - width,
            0,
            padded_height - height
        ),
        mode="reflect"
    )

    with torch.no_grad():
        output = llformer_model(
            input_tensor
        )

    output = torch.clamp(
        output,
        0,
        1
    )[:, :, :height, :width]

    output = img_as_ubyte(
        output.permute(
            0,
            2,
            3,
            1
        ).cpu().numpy()[0]
    )

    output_bgr = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )

    return cv2.resize(
        output_bgr,
        (
            original_width,
            original_height
        ),
        interpolation=cv2.INTER_LINEAR
    )


LLFORMER_OUTPUT_DIR = (
    WORK_DIR / "MS2_Road3_LLFormer_200"
)

LLFORMER_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

llformer_model = load_llformer()

try:
    for filename in tqdm(
        sample_files,
        desc="Pre-computing LLFormer images"
    ):
        output_path = (
            LLFORMER_OUTPUT_DIR / filename
        )

        if output_path.is_file():
            continue

        image_bgr = cv2.imread(
            str(RGB_DIR / filename)
        )

        if image_bgr is None:
            raise FileNotFoundError(
                f"Unable to read image: "
                f"{RGB_DIR / filename}"
            )

        enhanced = apply_llformer(
            image_bgr,
            llformer_model
        )

        if not cv2.imwrite(
            str(output_path),
            enhanced
        ):
            raise IOError(
                f"Unable to save LLFormer image: "
                f"{output_path}"
            )
finally:
    del llformer_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(
    f"LLFormer images saved to: "
    f"{LLFORMER_OUTPUT_DIR}"
)


Pre-computing LLFormer images: 100%|██████████| 200/200 [01:09<00:00,  2.87it/s]

LLFormer images saved to: /kaggle/working/MS2_Road3_LLFormer_200


## 6. Load DepthAnythingV2 Metric Outdoor Small


In [6]:
from transformers import (
    AutoImageProcessor,
    AutoModelForDepthEstimation
)

MODEL_ID = (
    "depth-anything/"
    "Depth-Anything-V2-Metric-Outdoor-Small-hf"
)

processor = AutoImageProcessor.from_pretrained(
    MODEL_ID
)

metric_model = (
    AutoModelForDepthEstimation
    .from_pretrained(MODEL_ID)
    .to(DEVICE)
    .eval()
)


def run_metric_depth(image_bgr):
    if image_bgr is None:
        raise ValueError(
            "Input image is None."
        )

    image_rgb = Image.fromarray(
        cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB
        )
    )

    inputs = processor(
        images=image_rgb,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = metric_model(**inputs)
        depth = outputs.predicted_depth

    depth = F.interpolate(
        depth.unsqueeze(1),
        size=image_rgb.size[::-1],
        mode="bicubic",
        align_corners=False
    ).squeeze().cpu().numpy()

    return depth.astype(np.float32)


print(
    f"Metric depth model loaded on {DEVICE}"
)


preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Metric depth model loaded on cuda


## 7. Run the Five-Pipeline Evaluation and Save Outputs

For every sampled image, the notebook saves:

- enhanced RGB images;
- raw metric-depth arrays (`.npy`);
- colourised metric-depth images (`.png`);
- per-image MAE, RMSE, AbsRel, and valid-pixel count.


In [7]:
RESULT_ROOT = (
    WORK_DIR / "MS2_Road3_metric_results"
)

OUTPUT_ROOT = RESULT_ROOT / "output"

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

PIPELINES = [
    "original",
    "gamma",
    "clahe",
    "msr",
    "llformer"
]

rows = []

for filename in tqdm(
    sample_files,
    desc="Evaluating MS2 Road3"
):
    image_bgr = cv2.imread(
        str(RGB_DIR / filename)
    )

    if image_bgr is None:
        raise FileNotFoundError(
            f"Unable to read image: "
            f"{RGB_DIR / filename}"
        )

    llformer_image = cv2.imread(
        str(
            LLFORMER_OUTPUT_DIR
            / filename
        )
    )

    if llformer_image is None:
        raise FileNotFoundError(
            f"Unable to read LLFormer image: "
            f"{LLFORMER_OUTPUT_DIR / filename}"
        )

    reference_depth = load_reference_depth(
        GT_DIR / filename
    )

    pipeline_images = {
        "original": image_bgr,
        "gamma": apply_gamma(image_bgr),
        "clahe": apply_clahe(image_bgr),
        "msr": apply_msr(image_bgr),
        "llformer": llformer_image
    }

    image_output_dir = (
        OUTPUT_ROOT / Path(filename).stem
    )

    rgb_output_dir = (
        image_output_dir / "rgb"
    )

    depth_output_dir = (
        image_output_dir / "depth"
    )

    rgb_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    depth_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    row = {
        "filename": filename
    }

    for method_name, pipeline_image in (
        pipeline_images.items()
    ):
        prediction = run_metric_depth(
            pipeline_image
        )

        metrics = compute_metrics(
            prediction,
            reference_depth,
            max_depth=80.0
        )

        if metrics is None:
            row[f"{method_name}_mae"] = np.nan
            row[f"{method_name}_rmse"] = np.nan
            row[f"{method_name}_absrel"] = np.nan
            row[
                f"{method_name}_valid_pixels"
            ] = 0
        else:
            for metric_name, value in (
                metrics.items()
            ):
                row[
                    f"{method_name}_{metric_name}"
                ] = value

        rgb_path = (
            rgb_output_dir
            / f"{method_name}.png"
        )

        depth_npy_path = (
            depth_output_dir
            / f"{method_name}.npy"
        )

        depth_png_path = (
            depth_output_dir
            / f"{method_name}.png"
        )

        if not cv2.imwrite(
            str(rgb_path),
            pipeline_image
        ):
            raise IOError(
                f"Unable to save RGB image: "
                f"{rgb_path}"
            )

        np.save(
            depth_npy_path,
            prediction
        )

        depth_visualisation = depth_to_vis(
            prediction
        )

        if not cv2.imwrite(
            str(depth_png_path),
            depth_visualisation
        ):
            raise IOError(
                f"Unable to save depth image: "
                f"{depth_png_path}"
            )

    rows.append(row)


per_image_df = pd.DataFrame(rows)

per_image_csv = (
    RESULT_ROOT
    / "MS2_Road3_metrics_per_image.csv"
)

per_image_df.to_csv(
    per_image_csv,
    index=False
)

print(
    f"Per-image metrics saved to: "
    f"{per_image_csv}"
)


Evaluating MS2 Road3: 100%|██████████| 200/200 [08:29<00:00,  2.55s/it]

Per-image metrics saved to: /kaggle/working/MS2_Road3_metric_results/MS2_Road3_metrics_per_image.csv


## 8. Create the Summary Table


In [8]:
summary_rows = []

for method_name in PIPELINES:
    summary_rows.append({
        "method": method_name,
        "mean_mae": per_image_df[
            f"{method_name}_mae"
        ].mean(),
        "mean_rmse": per_image_df[
            f"{method_name}_rmse"
        ].mean(),
        "mean_absrel": per_image_df[
            f"{method_name}_absrel"
        ].mean(),
        "n_images": len(per_image_df)
    })


summary_df = pd.DataFrame(
    summary_rows
)

summary_csv = (
    RESULT_ROOT
    / "MS2_Road3_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)

print(summary_df.to_string(index=False))
print(f"\nSummary saved to: {summary_csv}")


  method  mean_mae  mean_rmse  mean_absrel  n_images
original  8.498858  12.756272     0.306938       200
   gamma  9.466621  13.843932     0.347132       200
   clahe 12.117712  16.419339     0.468321       200
     msr  9.976561  14.341416     0.374372       200
llformer 11.856697  16.428044     0.452877       200

Summary saved to: /kaggle/working/MS2_Road3_metric_results/MS2_Road3_summary.csv


## 9. Package the Complete Results


In [9]:
# Include the sample list in the result folder.
shutil.copy2(
    sample_csv,
    RESULT_ROOT
    / sample_csv.name
)

archive_path = shutil.make_archive(
    str(
        WORK_DIR
        / "MS2_Road3_metric_results"
    ),
    "zip",
    str(RESULT_ROOT)
)

archive_size_mb = (
    Path(archive_path).stat().st_size
    / (1024 ** 2)
)

print("ZIP archive created:")
print(archive_path)
print(f"Archive size: {archive_size_mb:.2f} MB")


ZIP archive created:
/kaggle/working/MS2_Road3_metric_results.zip
Archive size: 2313.41 MB
